In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from scipy.optimize import least_squares

sys.path.insert(0, str(Path("../../nogse_pipeline/src")))
from models.model_fitting import M_ogse_tort, M_ogse_free
from plotting.publication.tc_param_vars import _roi_color_map, DEFAULT_BRAIN_MARKERS

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MASTER_PATH  = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR      = Path("../../analysis/brains/ogse_experiments/lab/fit_ogse_tort-tort_CC-N4-8-12-postcc-a5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS  = ["tra", "long"]
N_LIST      = [4, 8]
TD_LIST     = [76.0, 90.0, 120.0, 143.4, 210.0]  # ms
# ROI_LIST
#    None → All ROIS
#    ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC"] → CC
#    ["Left-Lateral-Ventricle", "Right-Lateral-Ventricle"] → Ventricles
ROI_LIST    = ["PostCC"]
G_COLUMN    = "g_thorsten"
G_CORRECTION_COLUMN = "grad_correction_factor"
Y_COLUMN    = "value"   # "value" (raw a.u.) or "value_norm" (normalised by S0)
D0_FIXED    = 3.2e-12       # m²/ms, fixed free-water diffusivity

#Fix T2_1 y T2_2 global values
T2_1 = 50.0 # ms, T2 of tort component 1
T2_2 = 50.0 # ms, T2 of tort component 2

# ALPHA_GLOBAL or TC_GLOBAL: scope for a shared parameter fit
#   None         → per-curve (one value per td, N, direction)
#   "td"         → one value per (subj, roi, direction, td), shared across N
#   "direction"  → one value per (subj, roi, direction), shared across td & N
#   "roi"        → one value per (subj, roi), shared across td, N & direction
#   "subj"       → one value per subject, shared across all roi, td, N & direction
#   "all"        → one value for all data

# ── Tortuosity factor α1 (tort component 1) ───────────────────────────────────
# ALPHA1_FIXED: same shapes as RN_FIXED-style nesting (None / "master" / float / nested dict).

ALPHA1_FIXED  = None

# ALPHA1_FIXED = {
#     "BRAIN": {
#         "AntCC": {"tra": 0., "long": 0.4},
#         "MidAntCC": {"tra": 0., "long": 0.4},
#         "CentralCC": {"tra": 0., "long": 0.4},
#         "MidPostCC": {"tra": 0., "long": 0.4},
#         "PostCC": {"tra": 0., "long": 0.4}
#     },
#     "LUDG": {
#         "AntCC": {"tra": 0., "long": 0.4},
#         "MidAntCC": {"tra": 0., "long": 0.4},
#         "CentralCC": {"tra": 0., "long": 0.4},
#         "MidPostCC": {"tra": 0., "long": 0.4},
#         "PostCC": {"tra": 0., "long": 0.4}
#     },
#     "MBBL": {
#         "AntCC": {"tra": 0., "long": 0.4},
#         "MidAntCC": {"tra": 0., "long": 0.4},
#         "CentralCC": {"tra": 0., "long": 0.4},
#         "MidPostCC": {"tra": 0., "long": 0.4},
#         "PostCC": {"tra": 0., "long": 0.4}
#     }
# }

ALPHA1_INIT   = 0.2
ALPHA1_BOUNDS = (0, 1)
ALPHA1_GLOBAL = None
ALPHA1_MASTER_COL = "alpha_macro"

# ── Tortuosity factor α2 (tort component 2) ───────────────────────────────────
# ALPHA2_FIXED: same shapes as ALPHA1_FIXED (None / "master" / float / nested dict).

ALPHA2_FIXED  = None

# ALPHA2_FIXED = {
#     "BRAIN": {
#         "AntCC": {"tra": 0.4, "long": 0.7},
#         "MidAntCC": {"tra": 0.4, "long": 0.7},
#         "CentralCC": {"tra": 0.4, "long": 0.7},
#         "MidPostCC": {"tra": 0.4, "long": 0.7},
#         "PostCC": {"tra": 0.4, "long": 0.7}
#     },
#     "LUDG": {
#         "AntCC": {"tra": 0.4, "long": 0.7},
#         "MidAntCC": {"tra": 0.4, "long": 0.7},
#         "CentralCC": {"tra": 0.4, "long": 0.7},
#         "MidPostCC": {"tra": 0.4, "long": 0.7},
#         "PostCC": {"tra": 0.4, "long": 0.7}
#     },
#     "MBBL": {
#         "AntCC": {"tra": 0.4, "long": 0.7},
#         "MidAntCC": {"tra": 0.4, "long": 0.7},
#         "CentralCC": {"tra": 0.4, "long": 0.7},
#         "MidPostCC": {"tra": 0.4, "long": 0.7},
#         "PostCC": {"tra": 0.4, "long": 0.7}
#     }
# }

ALPHA2_INIT   = 0.5
ALPHA2_BOUNDS = (0.0, 1)
ALPHA2_GLOBAL = None
ALPHA2_MASTER_COL = "alpha_macro"

# ── Mixing fractions f1, f2 ───────────────────────────────────────────────────
# Model:  S(G) = M0 · (f1·S_tort1(α1,G) + f2·S_tort2(α2,G))
# where   M0 = sqrt(S0² − RN²)  (Rician correction)
#
# F_TOTAL : float → f1 + f2 is constrained to F_TOTAL (f2 = F_TOTAL − f1, not fitted);
#           None  → f1 and f2 are independently free.
#
# F1_FIXED (no "master" option — no master-table column for f1):
#   None                                    → f1 is fitted
#   float                                   → same f1 for all subj/roi/direction/td
#   {subj: float}                           → fixed per subject
#   {subj: {roi: float}}                    → fixed per subject and roi
#   {subj: {roi: {direction: float}}}       → fixed per subject, roi, and direction
#   {subj: {roi: {direction: {td: float}}}} → fixed per subject, roi, direction, and td
#   Any (subj, roi, direction, td) combo absent from the dict is fitted freely —
#   this can mix fixed and free curves within the same GLOBAL joint fit.
#
# F1_GLOBAL: how many curves share the same f1 when it is free. (analogous to ALPHA1_GLOBAL / ALPHA2_GLOBAL)
#   None        → per-curve (one f1 per td, N, direction)
#   "td"        → one f1 per (subj, roi, direction, td), shared across N
#   "direction" → one f1 per (subj, roi, direction), shared across td & N
#   "roi"       → one f1 per (subj, roi), shared across td, N & direction
#   "subj"      → one f1 per subject
#   "all"       → one f1 for all data
F_TOTAL   = 1.0    # typically 1.0; set None to let f1 + f2 float freely
F1_FIXED  = None
F1_GLOBAL = "td"  # scope for shared f1

# ── Model name ────────────────────────────────────────────────────────────────
MODEL_NAME = 'tort-tort'

# ── Rician noise floor RN (signal units) ─────────────────────────────────────
# Signal model:  S(G) = sqrt( (M0·(f1·S_tort1 + f2·S_tort2))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#
#   None                        → no noise correction (RN = 0)
#   float                       → same RN for all subjects, td, and roi
#   {td: float}                 → fixed per td, same for all subjects and roi
#   {subj: float}               → fixed per subject, same for all td and roi
#   {subj: {td: float}}         → fixed per subject and td, same for all roi
#   {subj: {td: {roi: float}}}  → fixed per subject, td, and roi

RN_FIXED = {
    "BRAIN": {
        76.0:  {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
        90.0:  {"AntCC": 32, "MidAntCC": 38, "CentralCC": 36, "MidPostCC": 42, "PostCC": 36},
        120.0: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        143.4: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        210.0: {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
    },
    "LUDG": {
        90.0:  {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
        120.0: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        143.4: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        210.0: {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
    },
    "MBBL": {
        90.0:  {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
        120.0: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        143.4: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        210.0: {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
    },
}

# ── Noise-floor truncation ────────────────────────────────────────────────────
# Exclude data points where y < RN × (1 + G_TRUNCATE_EPSILON) from the fit.
# These points are dominated by Rician noise and distort the fitted parameters.
# Plots (Cell 6, Cell 8) are NOT affected — all original data points are shown.
#   None  → no truncation (keep all points, current behaviour)
#   float → e.g. 0.10 excludes points within 10 % above the noise floor
G_TRUNCATE_EPSILON = None

# ── Shared S0 across N values ─────────────────────────────────────────────────
# If |S0(Ni) − S0(Nj)| / S0_avg < S0_SHARE_THRESHOLD for all N pairs at the
# same (subj, roi, direction, td), use S0_avg for all N instead of individual S0.
# Prevents the normalization from amplifying small S0 fluctuations between N.
#   None  → each N uses its own S0 (current behaviour)
#   float → e.g. 0.02 → share S0 if they agree within 2 %
S0_SHARE_THRESHOLD = 0.05

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

missing_cols = [c for c in [G_COLUMN, Y_COLUMN, G_CORRECTION_COLUMN] if c is not None and c not in data.columns]
if missing_cols:
    raise KeyError(f"Missing column(s): {missing_cols}")

data["G_fit"] = data[G_COLUMN]
if G_CORRECTION_COLUMN is not None:
    data["G_fit"] = data["G_fit"] * data[G_CORRECTION_COLUMN]
data["y_raw"] = data["value"]   # raw signal — always kept to derive S0
data["y_fit"] = data[Y_COLUMN]

G_LABEL = f"Modulation gradient G [mT/m] ({G_COLUMN})"
Y_LABEL = f"{Y_COLUMN} [a.u.]"

if TD_LIST is not None:
    data = data[data.td_ms.isin(TD_LIST)]
if ROI_LIST is not None:
    data = data[data.roi.isin(ROI_LIST)]

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
print(f"Gradient column: {G_COLUMN}" + (f" × {G_CORRECTION_COLUMN}" if G_CORRECTION_COLUMN else ""))
print(f"Signal column:   {Y_COLUMN}")
data[["subj", "roi", "direction", "td_ms", "N", "G_fit", "y_fit"]]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT FUNCTIONS
#
# Signal model:  S(G) = sqrt( (M0·(f1·S_tort1 + f2·S_tort2))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#                S_tort1 = M_ogse_tort(α1, G)   [no tc]
#                S_tort2 = M_ogse_tort(α2, G)   [no tc]
#                f1 + f2 = F_TOTAL  if F_TOTAL is not None, else f1, f2 are independent
# ══════════════════════════════════════════════════════════════════════════════

def _resolve_rn(subj, td, roi=None):
    """Return fixed RN for (subj, td, roi). Returns 0.0 when RN_FIXED is None."""
    if RN_FIXED is None:
        return 0.0
    if isinstance(RN_FIXED, dict):
        first_key = next(iter(RN_FIXED))
        if isinstance(first_key, str):               # {subj: ...}
            subj_entry = RN_FIXED.get(subj)
            if subj_entry is None:
                return 0.0
            if isinstance(subj_entry, dict):         # {subj: {td: ...}}
                td_entry = subj_entry.get(float(td), subj_entry.get(td))
                if td_entry is None:
                    return 0.0
                if isinstance(td_entry, dict):       # {subj: {td: {roi: float}}}
                    return float(td_entry.get(roi, 0.0))
                return float(td_entry)               # {subj: {td: float}}
            return float(subj_entry)                 # {subj: float}
        return float(RN_FIXED.get(float(td), RN_FIXED.get(td, 0.0)))  # {td: float}
    return float(RN_FIXED)


def _m0_from_s0(s0_raw, rn):
    """M0 = sqrt(S0² − RN²). For value_norm returns M0/S0."""
    m0 = np.sqrt(max(float(s0_raw)**2 - float(rn)**2, 0.0))
    if Y_COLUMN == "value_norm":
        return m0 / float(s0_raw) if float(s0_raw) > 0 else 1.0
    return m0


def _get_master_value(subj, roi, direction, col):
    """Look up a fixed value from the master table for one (subj, roi, direction) group."""
    rows = df[(df.subj == subj) & (df.roi == roi) & (df.direction == direction)]
    if col in rows.columns and not rows[col].isna().all():
        return float(rows[col].dropna().iloc[0])
    raise KeyError(f"'{col}' not found or all-NaN for {subj}/{roi}/{direction}")


def _resolve_dict_fixed(fixed, subj, roi, direction, td):
    """Walk a nested dict keyed subj → roi → direction → td (each level optional —
    hitting a non-dict value fixes it for everything below). Returns the fixed
    float, or None if this combo isn't covered (→ fitted freely)."""
    val = fixed
    for key in (subj, roi, direction):
        if not isinstance(val, dict):
            return float(val)
        if key not in val:
            return None
        val = val[key]
    if isinstance(val, dict):
        td_f = float(td)
        if td_f in val:
            val = val[td_f]
        elif td in val:
            val = val[td]
        else:
            return None
    return None if isinstance(val, dict) else float(val)


def _resolve_alpha1(subj, roi, direction, td):
    """Return fixed α1, or None (→ fitted)."""
    if ALPHA1_FIXED is None:
        return None
    if ALPHA1_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA1_MASTER_COL)
    if isinstance(ALPHA1_FIXED, dict):
        return _resolve_dict_fixed(ALPHA1_FIXED, subj, roi, direction, td)
    return float(ALPHA1_FIXED)


def _resolve_alpha2(subj, roi, direction, td):
    """Return fixed α2, or None (→ fitted)."""
    if ALPHA2_FIXED is None:
        return None
    if ALPHA2_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA2_MASTER_COL)
    if isinstance(ALPHA2_FIXED, dict):
        return _resolve_dict_fixed(ALPHA2_FIXED, subj, roi, direction, td)
    return float(ALPHA2_FIXED)


def _resolve_f1(subj, roi, direction, td):
    """Return fixed f1, or None (→ fitted). No 'master' option — no master column for f1."""
    if F1_FIXED is None:
        return None
    if isinstance(F1_FIXED, dict):
        return _resolve_dict_fixed(F1_FIXED, subj, roi, direction, td)
    return float(F1_FIXED)


def _predict(td, G, N, alpha1, alpha2, f1, f2, M0, rn):
    """S(G) = sqrt( (M0·(f1·S_tort1 + f2·S_tort2))² + RN² )"""
    with np.errstate(over="ignore", invalid="ignore"):
        s1  = M_ogse_tort(td, G, N, td / N, alpha1, 1, D0_FIXED)
        s2  = M_ogse_tort(td, G, N, td / N, alpha2, 1, D0_FIXED)
        sig = M0 * (f1 * ( 1 / (f1 + f2*np.exp(td/T2_1 - td/T2_2) ) )  * s1 + f2 * ( 1 / (f2 + f1*np.exp(td/T2_2 - td/T2_1) ) ) * s2)
        return np.sqrt(sig**2 + rn**2) if rn != 0.0 else sig


def _param_stderr(result, param_idx):
    """1-σ error on params[param_idx] from the least-squares Jacobian."""
    dof = result.fun.size - result.x.size
    if dof <= 0 or result.x.size == 0 or param_idx >= result.x.size:
        return np.nan
    try:
        cov = np.linalg.pinv(result.jac.T @ result.jac) * (2.0 * result.cost / dof)
        return float(np.sqrt(max(float(cov[param_idx, param_idx]), 0.0)))
    except np.linalg.LinAlgError:
        return np.nan


def fit_curve(td, N, G, y, subj=None, roi=None, direction=None, s0=None):
    """Fit α1, α2, f1 for ONE (td, N, direction) curve.

    Returns (alpha1, alpha2, f1, f2, result, free).
    f1 is bounded in [0, F_TOTAL] when F_TOTAL is not None, else [0, 1].
    When F_TOTAL is not None: f2 = F_TOTAL − f1 (not a free parameter).
    When F_TOTAL is None: f2 is fitted independently in [0, 1].
    `free` is the ordered list of parameter names actually optimised (their
    position in `free` matches their index in `result.x`), needed by callers
    to extract per-parameter stderrs since which params are fixed can now
    vary per curve (dict-based *_FIXED).
    """
    rn = _resolve_rn(subj, td, roi)
    M0 = _m0_from_s0(s0, rn) if s0 is not None else float(y.max())

    al1_fv  = _resolve_alpha1(subj, roi, direction, td)
    al2_fv  = _resolve_alpha2(subj, roi, direction, td)
    f1_fv   = _resolve_f1(subj, roi, direction, td)
    f_total = float(F_TOTAL)  if F_TOTAL  is not None else None

    free, x0, lo, hi = [], [], [], []
    if al1_fv is None:
        free.append("alpha1")
        x0.append(ALPHA1_INIT); lo.append(ALPHA1_BOUNDS[0]); hi.append(ALPHA1_BOUNDS[1])
    if al2_fv is None:
        free.append("alpha2")
        x0.append(ALPHA2_INIT); lo.append(ALPHA2_BOUNDS[0]); hi.append(ALPHA2_BOUNDS[1])
    if f1_fv is None:
        free.append("f1")
        f1_hi = f_total if f_total is not None else 1.0
        x0.append(0.5 * f1_hi); lo.append(0.0); hi.append(f1_hi)
    if f_total is None:  # f2 is also free (independent of f1)
        free.append("f2")
        f2_init = 1.0 - (f1_fv if f1_fv is not None else 0.5)
        x0.append(max(0.0, f2_init)); lo.append(0.0); hi.append(1.0)

    def _unpack(params):
        idx = 0
        if "alpha1" in free:
            alpha1 = params[idx]; idx += 1
        else:
            alpha1 = al1_fv
        if "alpha2" in free:
            alpha2 = params[idx]; idx += 1
        else:
            alpha2 = al2_fv
        if "f1" in free:
            f1 = params[idx]; idx += 1
        else:
            f1 = f1_fv
        if f_total is not None:
            f2 = f_total - f1
        elif "f2" in free:
            f2 = params[idx]
        else:
            f2 = 1.0 - f1
        return alpha1, alpha2, f1, f2

    if not free:
        alpha1, alpha2, f1, f2 = _unpack([])
        fun = np.log(_predict(td, G, N, alpha1, alpha2, f1, f2, M0, rn)) - np.log(y)
        mock = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return alpha1, alpha2, f1, f2, mock, free

    def residuals(params):
        alpha1, alpha2, f1, f2 = _unpack(params)
        out = np.log(_predict(td, G, N, alpha1, alpha2, f1, f2, M0, rn)) - np.log(y)
        return np.where(np.isfinite(out), out, 1e6)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)
    alpha1, alpha2, f1, f2 = _unpack(result.x)
    return alpha1, alpha2, f1, f2, result, free


# ── Scope definitions for global fitting ─────────────────────────────────────
_SCOPE_RANK = {"all": 1, "subj": 2, "roi": 3, "direction": 4, "td": 5}
_SCOPE_COLS = {
    "all":       [],
    "subj":      ["subj"],
    "roi":       ["subj", "roi"],
    "direction": ["subj", "roi", "direction"],
    "td":        ["subj", "roi", "direction", "td_ms"],
}


def _group_scope():
    """Return (scope_name, group_cols) for the coarsest scope among the active
    global parameters.

    The outer split must gather every curve that ANY shared parameter spans,
    so it has to be at least as coarse as the coarsest requested _GLOBAL scope
    (e.g. ALPHA1_GLOBAL="direction" needs all td's of a direction in one call).
    Parameters whose own scope is finer than this outer split (e.g. F1_GLOBAL=
    "td" nested inside a "direction"-level call) are handled by giving them
    one shared index per distinct sub-key inside fit_group_global — NOT by
    shrinking the outer split down to the finest scope.
    """
    candidates = [s for s in (ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL) if s is not None]
    ranks = [_SCOPE_RANK[s] for s in candidates]
    coarsest = candidates[ranks.index(min(ranks))]
    return coarsest, _SCOPE_COLS[coarsest]


def fit_group_global(group_df):
    """Joint fit across all curves in group_df.

    Each of alpha1, alpha2, f1 is shared according to its OWN _GLOBAL
    scope: one fitted value per distinct key of _SCOPE_COLS[GLOBAL] present in
    group_df — not necessarily one value for the whole group_df. This lets
    different parameters use different sharing granularities within the same
    joint least_squares call (e.g. alpha1 shared over a whole direction while f1
    gets one value per td within that direction).

    Each parameter is resolved PER CURVE via its _resolve_* function: a curve
    is fixed (not optimised) when that combo is covered by its *_FIXED value
    (float / "master" / dict entry), and free otherwise. Since a dict may
    cover only some combos, fixed and free curves can be mixed within the
    same shared-scope group and the same joint least_squares call.

    Returns: (result, curves,
              per_al1s, per_al1_errs, per_al2s, per_al2_errs,
              per_f1s, per_f1_errs, per_f2s, M0s)
    """
    f_total = float(F_TOTAL) if F_TOTAL is not None else None

    curves = []
    for (subj, roi, direction), dir_grp in group_df.groupby(["subj", "roi", "direction"]):
        for td in sorted(float(t) for t in dir_grp.td_ms.unique()):
            sub = dir_grp[np.isclose(dir_grp.td_ms.astype(float), td)]
            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                s0_raw = float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                rn = _resolve_rn(subj, td, roi)
                curves.append(dict(
                    subj=subj, roi=roi, direction=direction,
                    td=td, N=int(N), S0=s0_raw,
                    G=s.G_fit.values, y=s.y_fit.values,
                    M0=_m0_from_s0(s0_raw, rn),
                    rn=rn,
                    al1_fv=_resolve_alpha1(subj, roi, direction, td),
                    al2_fv=_resolve_alpha2(subj, roi, direction, td),
                    f1_fv=_resolve_f1(subj, roi, direction, td),
                ))

    M0s = {(c["subj"], c["roi"], c["direction"], c["td"], c["N"]): c["M0"] for c in curves}

    def _curve_key(scope, c):
        lut = {"subj": c["subj"], "roi": c["roi"], "direction": c["direction"], "td_ms": c["td"]}
        return tuple(lut[col] for col in _SCOPE_COLS[scope])

    x0, lo, hi = [], [], []

    def _add_param(fixed_per_curve, global_scope, init, lo_b, hi_b):
        """Per curve: if fixed_per_curve[i] is not None, that curve is pinned
        (no x0 entry). Otherwise it shares an x0 entry with other free curves
        that have the same global_scope key (or gets its own when global_scope
        is None). Returns a per-curve list of ("fixed", value) / ("free", idx)."""
        key_to_idx = {}
        entries = []
        for i, c in enumerate(curves):
            fv = fixed_per_curve[i]
            if fv is not None:
                entries.append(("fixed", fv))
                continue
            key = _curve_key(global_scope, c) if global_scope is not None else i
            if key not in key_to_idx:
                key_to_idx[key] = len(x0)
                x0.append(init); lo.append(lo_b); hi.append(hi_b)
            entries.append(("free", key_to_idx[key]))
        return entries

    al1_entries = _add_param([c["al1_fv"] for c in curves], ALPHA1_GLOBAL,
                              ALPHA1_INIT, ALPHA1_BOUNDS[0], ALPHA1_BOUNDS[1])
    al2_entries = _add_param([c["al2_fv"] for c in curves], ALPHA2_GLOBAL,
                              ALPHA2_INIT, ALPHA2_BOUNDS[0], ALPHA2_BOUNDS[1])

    f1_hi = f_total if f_total is not None else 1.0
    f1_entries = _add_param([c["f1_fv"] for c in curves], F1_GLOBAL, 0.5 * f1_hi, 0.0, f1_hi)

    f2_entries = None
    if f_total is None:  # f2 always free, per curve, when F_TOTAL is None
        f2_entries = _add_param([None] * len(curves), None, 0.5, 0.0, 1.0)

    def _curve_params(params, i, c):
        kind, val = al1_entries[i]
        alpha1 = params[val] if kind == "free" else val

        kind, val = al2_entries[i]
        alpha2 = params[val] if kind == "free" else val

        kind, val = f1_entries[i]
        f1 = params[val] if kind == "free" else val

        if f_total is not None:
            f2 = f_total - f1
        elif f2_entries is not None:
            f2 = params[f2_entries[i][1]]
        else:
            f2 = 1.0 - f1

        return alpha1, alpha2, f1, f2

    if not x0:
        # Every curve is fixed for every parameter (only reachable when
        # f_total is not None — otherwise f2 always adds x0 entries).
        f1_vals, f2_vals, fun_parts = [], [], []
        for c in curves:
            f1_val = c["f1_fv"]
            f2_val = f_total - f1_val
            f1_vals.append(f1_val); f2_vals.append(f2_val)
            fun_parts.append(np.log(_predict(c["td"], c["G"], c["N"],
                                              c["al1_fv"], c["al2_fv"],
                                              f1_val, f2_val, c["M0"], c["rn"])) - np.log(c["y"]))
        fun = np.concatenate(fun_parts)
        result = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return (result, curves,
                [c["al1_fv"] for c in curves], [np.nan] * len(curves),
                [c["al2_fv"] for c in curves], [np.nan] * len(curves),
                f1_vals, [np.nan] * len(curves),
                f2_vals, M0s)

    def residuals(params):
        res = []
        for i, c in enumerate(curves):
            alpha1, alpha2, f1, f2 = _curve_params(params, i, c)
            out = np.log(_predict(c["td"], c["G"], c["N"], alpha1, alpha2, f1, f2, c["M0"], c["rn"])) - np.log(c["y"])
            res.append(np.where(np.isfinite(out), out, 1e6))
        return np.concatenate(res)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)

    per_al1s, per_al1e = [], []
    per_al2s, per_al2e = [], []
    per_f1s,  per_f1e  = [], []
    per_f2s            = []

    for i, c in enumerate(curves):
        alpha1, alpha2, f1, f2 = _curve_params(result.x, i, c)
        per_al1s.append(alpha1); per_al2s.append(alpha2)
        per_f1s.append(f1);      per_f2s.append(f2)

        kind, val = al1_entries[i]
        per_al1e.append(_param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = al2_entries[i]
        per_al2e.append(_param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = f1_entries[i]
        per_f1e.append(_param_stderr(result, val) if kind == "free" else np.nan)

    return (result, curves,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s, M0s)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT LOOP  —  runs all (subj, roi, dir, td, N) groups
# ══════════════════════════════════════════════════════════════════════════════

rows      = []
fit_store = {}


def _scalar_err(result, pidx):
    return _param_stderr(result, pidx) if pidx is not None else np.nan


_any_global = any(v is not None for v in [ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL])

if not _any_global:
    # ── Per-curve fit: one (α1, α2, f1) per (subj, roi, dir, td, N) ──
    for (subj, roi, direction), grp in data.groupby(["subj", "roi", "direction"]):
        tds = sorted(float(t) for t in grp.td_ms.unique())
        fit_store[(subj, roi, direction)] = {"tds": tds, "fits": {td: {} for td in tds}}

        for td in tds:
            sub = grp[np.isclose(grp.td_ms.astype(float), td)]

            # ── Pre-pass: collect S0 per N, optionally share ─────────────────
            s0_per_N = {}
            for _N in N_LIST:
                _s = sub[sub.N == _N].sort_values("G_fit")
                if not _s.empty:
                    s0_per_N[_N] = float(
                        _s.loc[_s.G_fit.abs() == _s.G_fit.abs().min(), "y_raw"].iloc[0]
                    )
            if S0_SHARE_THRESHOLD is not None and len(s0_per_N) >= 2:
                _s0_vals = list(s0_per_N.values())
                _s0_avg  = float(np.mean(_s0_vals))
                _max_dev = max(abs(v - _s0_avg) for v in _s0_vals)
                if _s0_avg > 0 and _max_dev / _s0_avg < S0_SHARE_THRESHOLD:
                    s0_per_N = {_N: _s0_avg for _N in s0_per_N}

            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                G, y = s.G_fit.values, s.y_fit.values
                s0_raw = s0_per_N.get(
                    N,
                    float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                )

                # ── Truncate noise-floor points (fit only, plots unchanged) ──
                G_fit, y_fit = G, y
                if G_TRUNCATE_EPSILON is not None:
                    _rn_trunc = _resolve_rn(subj, td, roi)
                    if _rn_trunc > 0:
                        _keep = y > _rn_trunc * (1.0 + G_TRUNCATE_EPSILON)
                        G_fit = G[_keep]
                        y_fit = y[_keep]
                if len(G_fit) == 0:
                    continue

                alpha1, alpha2, f1, f2, result, free = fit_curve(
                    td, N, G_fit, y_fit, subj=subj, roi=roi, direction=direction, s0=s0_raw
                )
                al1_pi = free.index("alpha1") if "alpha1" in free else None
                al2_pi = free.index("alpha2") if "alpha2" in free else None
                f1_pi  = free.index("f1")     if "f1"     in free else None
                rn = _resolve_rn(subj, td, roi)
                M0 = _m0_from_s0(s0_raw, rn)

                print(
                    f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  N={N}  "
                    f"α1={alpha1:.4f}  α2={alpha2:.4f}  "
                    f"f1={f1:.3f}  M0={M0:.4f}  RN={rn:.2f}  cost={result.cost:.3e}"
                )
                fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                    G=G, y=y, alpha1=alpha1, alpha2=alpha2,
                    f1=f1, f2=f2, M0=M0, RN=rn, S0=s0_raw,
                )
                rows.append(dict(
                    subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                    alpha1=alpha1, alpha1_err=_scalar_err(result, al1_pi),
                    alpha2=alpha2, alpha2_err=_scalar_err(result, al2_pi),
                    f1=f1, f1_err=_scalar_err(result, f1_pi), f2=f2,
                    M0=M0, D0_m2ms=D0_FIXED, RN=rn,
                    model_name=MODEL_NAME,
                    g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                    cost=float(result.cost), success=bool(result.success),
                ))

else:
    # ── Global fit: shared params over the specified scope ────────────────────
    scope_name, scope_cols = _group_scope()
    groups = data.groupby(scope_cols) if scope_cols else [("all", data)]

    for group_key, group_df in groups:
        (result, curves,
         per_al1s, per_al1e, per_al2s, per_al2e,
         per_f1s, per_f1e, per_f2s, M0s) = fit_group_global(group_df)

        key_str = "  ".join(str(k) for k in group_key) if isinstance(group_key, tuple) else str(group_key)
        print(f"[{key_str}]  cost={result.cost:.3e}")

        for (subj, roi, direction), sub_grp in group_df.groupby(["subj", "roi", "direction"]):
            entry = fit_store.setdefault((subj, roi, direction), {"tds": [], "fits": {}})
            for _td in sorted(float(t) for t in sub_grp.td_ms.unique()):
                if _td not in entry["fits"]:
                    entry["tds"].append(_td)
                    entry["fits"][_td] = {}
            entry["tds"].sort()

        for c, al1, al1e, al2, al2e, f1, f1e, f2 in zip(
            curves,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s,
        ):
            subj, roi, direction, td, N = c["subj"], c["roi"], c["direction"], c["td"], c["N"]
            M0 = M0s[(subj, roi, direction, td, N)]
            fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                G=c["G"], y=c["y"], alpha1=al1, alpha2=al2,
                f1=f1, f2=f2, M0=M0, RN=c["rn"], S0=c["S0"],
            )
            rows.append(dict(
                subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                alpha1=al1, alpha1_err=al1e, alpha2=al2, alpha2_err=al2e,
                f1=f1, f1_err=f1e, f2=f2,
                M0=M0, D0_m2ms=D0_FIXED, RN=c["rn"],
                model_name=MODEL_NAME,
                g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                cost=float(result.cost), success=bool(result.success),
            ))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS TABLE
# Columns: subj, roi, direction, td_ms, N,
#          alpha1, alpha1_err, alpha2, alpha2_err,
#          f1, f1_err, f2, M0, D0_m2ms, RN, cost, ...
# ══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(rows)
results_df.to_excel(OUT_DIR / "fit_results.xlsx", index=False)
print(f"Saved {len(results_df)} rows → {OUT_DIR / 'fit_results.xlsx'}")
results_df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT FITS — raw signal with Rician noise correction
# One figure per (subj, roi, dir): data points + fitted S(G) per (td, N)
# Top row:    S(G) = sqrt( (M0·(f1·S_tort1 + f2·S_tort2))² + RN² )
# Bottom row: signal contrast  ΔS = S(N_hi) − S(N_lo)  vs G
#             restricted to min(max_G_N_hi, max_G_N_lo)
# ══════════════════════════════════════════════════════════════════════════════

G_plot = np.linspace(0, float(data.G_fit.max()), 300)
N_hi, N_lo = N_LIST[-1], N_LIST[0]

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        title_lines = [f"td = {td:.1f} ms"]
        y_hat_per_N = {}
        max_G_per_N = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            if fit is None:
                continue
            max_G_per_N[N] = float(fit["G"].max())
            ax.scatter(fit["G"], fit["y"], color=color, s=20, zorder=3, label=f"N={N} data")
            y_hat = _predict(td, G_plot, N,
                             fit["alpha1"], fit["alpha2"],
                             fit["f1"], fit["f2"], fit["M0"], fit["RN"])
            y_hat_per_N[N] = y_hat
            ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit")
            title_lines.append(
                f"α1={fit['alpha1']:.3f}  α2={fit['alpha2']:.3f}  "
                f"f1={fit['f1']:.3f}  RN={fit['RN']:.1f}  (N={N})"
            )
        ax.set_title("\n".join(title_lines), fontsize=6)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            G_diff = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
            fit_hi, fit_lo = fits_td[N_hi], fits_td[N_lo]
            diff_hi = _predict(td, G_diff, N_hi, fit_hi["alpha1"], fit_hi["alpha2"],
                               fit_hi["f1"], fit_hi["f2"], fit_hi["M0"], fit_hi["RN"])
            diff_lo = _predict(td, G_diff, N_lo, fit_lo["alpha1"], fit_lo["alpha2"],
                               fit_lo["f1"], fit_lo["f2"], fit_lo["M0"], fit_lo["RN"])
            dax.plot(G_diff, diff_hi - diff_lo, color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(Y_LABEL)
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Fit plots saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT parameters vs td  —  all subjects, grouped by ROI and direction
# One figure per (param, direction); one panel per ROI
# Colour = subject (lighter → darker within ROI palette), linestyle = N
# ══════════════════════════════════════════════════════════════════════════════

rois     = sorted(results_df.roi.unique())
subjects = sorted(results_df.subj.unique())
roi_colors   = _roi_color_map(rois)
_N_ls        = ["-", "--", "-.", ":"]
N_linestyles = {N: ls for N, ls in zip(N_LIST, _N_ls)}

for direction in DIRECTIONS:
    sub = results_df[results_df.direction == direction]
    for param, ylabel in [
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(1, len(rois), figsize=(4 * len(rois), 4), sharey=True)
        axes = np.atleast_1d(axes)

        for ax, roi in zip(axes, rois):
            base_color = roi_colors[roi]
            cmap = mcolors.LinearSegmentedColormap.from_list("", ["#cccccc", base_color])
            for i, subj in enumerate(subjects):
                shade  = cmap(0.3 + 0.7 * i / max(1, len(subjects) - 1))
                marker = DEFAULT_BRAIN_MARKERS[i % len(DEFAULT_BRAIN_MARKERS)]
                for N in N_LIST:
                    g = (
                        sub[(sub.roi == roi) & (sub.subj == subj) & (sub.N == N)]
                        .drop_duplicates("td_ms").sort_values("td_ms")
                    )
                    if g.empty:
                        continue
                    label = subj if N == N_LIST[0] else None
                    ax.scatter(g.td_ms, g[param], color=shade, marker=marker, s=60, zorder=3, label=label)
                    ax.plot(g.td_ms, g[param], color=shade, linewidth=0.8,
                            linestyle=N_linestyles[N])
            ax.set_title(roi, fontsize=9)
            ax.set_xlabel("td [ms]")
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)

        axes[0].set_ylabel(ylabel)
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, fontsize=7, loc="upper right", ncol=1)
        n_handles = [
            mlines.Line2D([], [], color="gray", linestyle=N_linestyles[N], label=f"N={N}")
            for N in N_LIST
        ]
        axes[-1].legend(handles=n_handles, fontsize=7, loc="lower right")
        fig.suptitle(f"{param} vs td — direction: {direction}", fontsize=10)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{direction}.png", dpi=120)
        plt.close(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT NORMALIZED FITS  —  M0·(f1·S_tort1 + f2·S_tort2) with no Rician correction
# Useful to compare model shape without amplitude effects.
# Data is corrected: sqrt(|y² − RN²|) / M0 → compared to f1·S_tort1 + f2·S_tort2
# Bottom row diff restricted to min(max_G_N_hi, max_G_N_lo)
# ══════════════════════════════════════════════════════════════════════════════

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        y_hat_per_N = {}
        max_G_per_N = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            title_lines = [f"td = {td:.1f} ms"]
            if fit is None:
                continue
            max_G_per_N[N] = float(fit["G"].max())
            M0 = fit["M0"] if fit["M0"] != 0.0 else 1.0
            RN = fit["RN"]
            y_corr = np.sqrt(np.abs(fit["y"]**2 - RN**2))*np.sign(fit["y"]**2 - RN**2) / M0
            ax.scatter(fit["G"], y_corr, color=color, s=20, zorder=3, label=f"N={N} data corr")
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_tort(td, G_plot, N, td / N, fit["alpha1"], 1.0, D0_FIXED)
                s2 = M_ogse_tort(td, G_plot, N, td / N, fit["alpha2"], 1.0, D0_FIXED)
                sig_norm = fit["f1"] * ( 1 / (fit["f2"] + fit["f1"]*np.exp(td/T2_1 - td/T2_2) ) ) * s1 + fit["f2"] * ( 1 / (fit["f2"] + fit["f1"]*np.exp(td/T2_2 - td/T2_1) ) ) * s2 
            y_hat_per_N[N] = sig_norm
            ax.plot(G_plot, sig_norm, color=color, label=f"N={N} model")
            title_lines.append(
                f"α1={fit['alpha1']:.3f}  α2={fit['alpha2']:.3f}  "
                f"f1={fit['f1']:.3f}  RN={fit['RN']:.1f}  (N={N})"
            )
            ax.set_yscale('log')
        ax.set_title("\n".join(title_lines), fontsize=6)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            G_diff = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
            fit_hi, fit_lo = fits_td[N_hi], fits_td[N_lo]
            with np.errstate(over="ignore", invalid="ignore"):
                s1_hi = M_ogse_tort(td, G_diff, N_hi, td / N_hi, fit_hi["alpha1"], 1.0, D0_FIXED)
                s2_hi = M_ogse_tort(td, G_diff, N_hi, td / N_hi, fit_hi["alpha2"], 1.0, D0_FIXED)
                s1_lo = M_ogse_tort(td, G_diff, N_lo, td / N_lo, fit_lo["alpha1"], 1.0, D0_FIXED)
                s2_lo = M_ogse_tort(td, G_diff, N_lo, td / N_lo, fit_lo["alpha2"], 1.0, D0_FIXED)
            diff_hi = fit_hi["f1"] * s1_hi + fit_hi["f2"] * s2_hi
            diff_lo = fit_lo["f1"] * s1_lo + fit_lo["f2"] * s2_lo
            dax.plot(G_diff, diff_hi - diff_lo, color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(f"{Y_COLUMN} / M0 [a.u.]")
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}  |  f1·S_tort1+f2·S_tort2  (clean model)", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_norm_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Normalized model plots saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT α1, α2, f1 and M0 vs td  —  per subject
# Rows = ROIs, columns = directions, colour/linestyle = N
# Shaded band = ±1σ from the Jacobian covariance
# ══════════════════════════════════════════════════════════════════════════════

subjects_plot = sorted(results_df.subj.unique())
rois_plot     = sorted(results_df.roi.unique())
Ns_plot       = sorted(results_df.N.unique())
dirs_plot     = [d for d in ["long", "tra"] if d in results_df.direction.unique()]
_N_ls_p       = ["-", "--", "-.", ":"]
N_ls          = {N: _N_ls_p[i % len(_N_ls_p)] for i, N in enumerate(Ns_plot)}

for subj in subjects_plot:
    for param, ylabel in [
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x    = g.td_ms.to_numpy(dtype=float)
                        yv   = g[param].to_numpy(dtype=float)
                        yerr = g[err_col].to_numpy(dtype=float)
                        ok   = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr)
                        if np.any(ok):
                            ax.fill_between(x[ok], yv[ok] - yerr[ok], yv[ok] + yerr[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel}", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{subj}.png", dpi=120)
        plt.close(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT α1, α2, f1 and M0 vs td  —  per subject, log-scale y-axis
# Same layout as previous cell; useful to spot order-of-magnitude variation
# ══════════════════════════════════════════════════════════════════════════════

for subj in subjects_plot:
    for param, ylabel in [
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x     = g.td_ms.to_numpy(dtype=float)
                        yv    = g[param].to_numpy(dtype=float)
                        yerr  = g[err_col].to_numpy(dtype=float)
                        lower = np.maximum(yv - yerr, np.finfo(float).tiny)
                        upper = yv + yerr
                        ok    = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr) & (yv > 0) & (upper > 0)
                        if np.any(ok):
                            ax.fill_between(x[ok], lower[ok], upper[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                ax.set_yscale("log")
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel} (log)", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td  (log scale)", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_log_{subj}.png", dpi=120)
        plt.close(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT SIGNAL CONTRAST  ΔS = S(N_hi) − S(N_lo)  vs G, Ld, Lcf, lcf
# Per subject: rows = ROIs, columns = directions, colour = td
# Contrast computed over the common G range: min(max_G_N_hi, max_G_N_lo)
# "_raw"  uses the full model:  sqrt( (M0·(f1·S_tort1+f2·S_tort2))² + RN² )
# "_corr" uses the clean model: M0·(f1·S_tort1+f2·S_tort2)  (no Rician term)
# ══════════════════════════════════════════════════════════════════════════════

td_cmap    = plt.cm.viridis
N_hi, N_lo = N_LIST[-1], N_LIST[0]
G_plot     = np.linspace(0, float(data.G_fit.max()), 300)
PEAK_GAMMA = 267.5221900   # rad / (ms · mT), proton gyromagnetic ratio
D0_PLOT    = D0_FIXED

def _x_for_G(G_arr, td_ms, xvar):
    if xvar == "G":
        return G_arr.copy()
    D0, gamma = float(D0_PLOT), float(PEAK_GAMMA)
    l_d = np.sqrt(D0 * float(td_ms))
    l_G = np.full_like(G_arr, np.nan)
    valid = G_arr > 0
    l_G[valid] = (D0 / (gamma * G_arr[valid])) ** (1.0 / 3.0)
    Ld  = l_d / l_G
    Lcf = 1.5 ** 0.25 / np.sqrt(Ld)
    lcf = Lcf * l_G * 1e6
    if xvar == "Ld":  return Ld
    if xvar == "Lcf_a": return Lcf
    if xvar == "lcf": return lcf
    raise ValueError(xvar)

X_AXES = [
    ("G",   G_LABEL),
    ("Ld",  "Ld (dimensionless)"),
    ("Lcf_a", "Lcf (dimensionless)"),
    ("lcf", "lcf [µm]"),
]

X_LIMITS = {
    "Lcf_a": (0.5, 1.4),
    "lcf": (0, 35),
}

# ── precompute contrasts ───────────────────────────────────────────────────────
raw_contrast_store  = {}
corr_contrast_store = {}
contrast_G_store    = {}   # G_common per (subj, roi, direction, td)
free_contrast_store = {}   # free (unrestricted) model contrast, same D0 as other models

for (subj, roi, direction), store in fit_store.items():
    for td in store["tds"]:
        fits_td = store["fits"][td]
        max_G_per_N = {
            N: float(fits_td[N]["G"].max())
            for N in N_LIST if fits_td.get(N) is not None
        }
        if N_hi not in max_G_per_N or N_lo not in max_G_per_N:
            continue
        G_common = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
        key = (subj, roi, direction, td)
        raw_per_N  = {}
        corr_per_N = {}
        for N in N_LIST:
            fit = fits_td.get(N)
            if fit is None:
                continue
            rn = fit.get("RN", 0.0)
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_tort(td, G_common, N, td / N, fit["alpha1"], 1, D0_FIXED)
                s2 = M_ogse_tort(td, G_common, N, td / N, fit["alpha2"], 1, D0_FIXED)
                bimodal  = fit["f1"] * ( 1 / (fit["f2"] + fit["f1"]*np.exp(td/T2_1 - td/T2_2) ) ) * s1 + fit["f2"] * ( 1 / (fit["f2"] + fit["f1"]*np.exp(td/T2_2 - td/T2_1) ) ) * s2
                raw_per_N[N]  = np.sqrt(bimodal**2 + (rn / fit["M0"])**2) if rn != 0.0 else bimodal
                corr_per_N[N] = bimodal
        if N_hi in raw_per_N and N_lo in raw_per_N:
            raw_contrast_store[key]  = raw_per_N[N_hi]  - raw_per_N[N_lo]
            contrast_G_store[key]    = G_common
        if N_hi in corr_per_N and N_lo in corr_per_N:
            corr_contrast_store[key] = corr_per_N[N_hi] - corr_per_N[N_lo]
        with np.errstate(over="ignore", invalid="ignore"):
            sig_hi = M_ogse_free(td, G_common, N_hi, td / N_hi, 1, D0_FIXED)
            sig_lo = M_ogse_free(td, G_common, N_lo, td / N_lo, 1, D0_FIXED)
        free_contrast_store[key] = sig_hi - sig_lo


def _plot_contrast(cstore, xvar, xlabel, subj, suffix, title_tag, free_store=None):
    rois_subj = sorted(set(k[1] for k in cstore if k[0] == subj))
    dirs_subj = [d for d in DIRECTIONS if any(k[2] == d for k in cstore if k[0] == subj)]
    n_rois, n_dirs = len(rois_subj), len(dirs_subj)
    if n_rois == 0 or n_dirs == 0:
        return
    all_tds_subj = sorted(set(k[3] for k in cstore if k[0] == subj))
    colors_leg   = [td_cmap(i / max(1, len(all_tds_subj) - 1)) for i in range(len(all_tds_subj))]
    leg_handles  = [plt.Line2D([0], [0], color=c, linewidth=1.5) for c in colors_leg]
    leg_labels   = [f"td={td:.0f} ms" for td in all_tds_subj]
    td_color_map = {td: c for td, c in zip(all_tds_subj, colors_leg)}
    if free_store is not None:
        leg_handles.append(plt.Line2D([0], [0], color="lightgrey", linestyle=":", linewidth=1.5))
        leg_labels.append("free")
    fig, axes = plt.subplots(n_rois, n_dirs, figsize=(4 * n_dirs, 3 * n_rois), squeeze=False, sharey="row")
    for i_roi, roi in enumerate(rois_subj):
        for i_dir, direction in enumerate(dirs_subj):
            ax = axes[i_roi, i_dir]
            tds_here = sorted(k[3] for k in cstore
                              if k[0] == subj and k[1] == roi and k[2] == direction)
            if not tds_here:
                ax.set_visible(False)
                continue
            for td in tds_here:
                key = (subj, roi, direction, td)
                G_arr = contrast_G_store.get(key)
                if G_arr is None:
                    continue
                ax.plot(_x_for_G(G_arr, td, xvar), cstore[key],
                        color=td_color_map[td], linewidth=1.2)
                if free_store is not None and key in free_store:
                    ax.plot(_x_for_G(G_arr, td, xvar), free_store[key],
                            color="lightgrey", linestyle=":", linewidth=1.2)
            ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
            if i_roi == 0:
                ax.set_title(direction, fontsize=9)
            if i_dir == 0:
                ax.set_ylabel(f"{roi}\nΔS (N={N_hi}−N={N_lo})", fontsize=7)
            ax.set_xlabel(xlabel, fontsize=7)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            ax.tick_params(labelsize=6)
            if xvar in X_LIMITS:
                ax.set_xlim(*X_LIMITS[xvar])
            ax.tick_params(axis="y", labelleft=True)
    fig.suptitle(f"{subj} — contrast ΔS (N={N_hi}−N={N_lo}) vs {xvar}  [{title_tag}]", fontsize=10)
    fig.legend(leg_handles, leg_labels,
               loc="upper center", ncol=len(leg_labels), fontsize=7, frameon=False,
               bbox_to_anchor=(0.5, 0.96), bbox_transform=fig.transFigure)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(OUT_DIR / f"contrast_vs_{xvar}_{subj}_{suffix}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)


all_subjs = sorted(set(k[0] for k in raw_contrast_store))
for xvar, xlabel in X_AXES:
    for subj in all_subjs:
        _plot_contrast(raw_contrast_store,  xvar, xlabel, subj, "raw",  "fit w/ RN")
        _plot_contrast(corr_contrast_store, xvar, xlabel, subj, "corr", "clean model", free_store=free_contrast_store)

print("Per-subject contrast figures saved (_raw and _corr).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTRAST CURVE FITTING — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# Fitting target: corr_contrast_store[(subj,roi,dir,td)]  — model contrast (M0=1)
#                 evaluated on contrast_G_store[(subj,roi,dir,td)]  (dense G_common)
# M0 = 1 always.  D0 always fixed.  Only tc and alpha are free or pinned.

CF_OUT_SUBDIR = "contrast_fits_corr"  # sub-folder inside OUT_DIR

# ── Free model ────────────────────────────────────────────────────────────────
CF_FREE_D0_FIXED   = D0_FIXED   # m²/ms

# ── Rest model ────────────────────────────────────────────────────────────────
CF_REST_TC_INIT   = 2.0         # ms
CF_REST_TC_FIXED  = None        # None → fitted; float → pinned
CF_REST_TC_BOUNDS = (0.05, 1000.0)
CF_REST_D0_FIXED  = D0_FIXED

# ── Tort model ────────────────────────────────────────────────────────────────
CF_TORT_ALPHA_INIT       = 0.1
CF_TORT_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_TORT_ALPHA_BOUNDS     = (0.0, 1.0)
CF_TORT_ALPHA_MASTER_COL = "alpha_macro"
CF_TORT_D0_FIXED         = D0_FIXED

# ── Mixed model ───────────────────────────────────────────────────────────────
CF_MIXED_TC_INIT          = 2.0
CF_MIXED_TC_FIXED         = None
CF_MIXED_TC_BOUNDS        = (0.05, 1000.0)
CF_MIXED_ALPHA_INIT       = 0.5
CF_MIXED_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_MIXED_ALPHA_BOUNDS     = (0.0, 1.0)
CF_MIXED_ALPHA_MASTER_COL = "alpha_macro"
CF_MIXED_D0_FIXED         = D0_FIXED

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT CONTRAST CURVES — 4 SINGLE-COMPONENT MODELS
# Fitting target: corr_contrast_store (M0-normalised, noise-free model contrast)
# M0 = 1 always.  D0 fixed per model.  tc fitted in log-space; alpha linear.
# ══════════════════════════════════════════════════════════════════════════════

from models.model_fitting import (
    OGSE_contrast_vs_g_free, OGSE_contrast_vs_g_rest,
    OGSE_contrast_vs_g_tort, OGSE_contrast_vs_g_mixed,
)

(OUT_DIR / CF_OUT_SUBDIR).mkdir(parents=True, exist_ok=True)

contrast_fit_store = {}
cf_rows = []

for key in sorted(corr_contrast_store):
    subj, roi, direction, td = key
    G_fit = contrast_G_store[key]
    C_fit = corr_contrast_store[key]

    # ── circle markers: corr contrast values at actual data G points ──────────
    fits_td   = fit_store.get((subj, roi, direction), {}).get("fits", {}).get(td, {})
    G_hi_data = fits_td[N_hi]["G"] if N_hi in fits_td else np.array([])
    G_lo_data = fits_td[N_lo]["G"] if N_lo in fits_td else np.array([])
    C_at_Ghi  = np.interp(G_hi_data, G_fit, C_fit)
    C_at_Glo  = np.interp(G_lo_data, G_fit, C_fit)

    entry = {
        "G_common": G_fit, "C_corr": C_fit,
        "G_hi": G_hi_data, "C_hi": C_at_Ghi,
        "G_lo": G_lo_data, "C_lo": C_at_Glo,
    }

    # ── free model (no free physics params) ───────────────────────────────────
    _D0_f   = CF_FREE_D0_FIXED
    _C_free = OGSE_contrast_vs_g_free(td, G_fit, G_fit, N_hi, N_lo, 1.0, _D0_f)
    entry["free"] = {
        "D0": _D0_f, "C_curve": _C_free,
        "cost": float(0.5 * np.sum((_C_free - C_fit) ** 2)),
    }

    # ── rest model ────────────────────────────────────────────────────────────
    _D0_r = CF_REST_D0_FIXED
    if CF_REST_TC_FIXED is not None:
        _tc_r = float(CF_REST_TC_FIXED)
    else:
        def _res_rest(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_r):
            r = OGSE_contrast_vs_g_rest(_td, _G, _G, N_hi, N_lo, np.exp(p[0]), 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_rest = least_squares(
            _res_rest, [np.log(CF_REST_TC_INIT)],
            bounds=([np.log(CF_REST_TC_BOUNDS[0])], [np.log(CF_REST_TC_BOUNDS[1])]),
            method="trf", max_nfev=10000,
        )
        _tc_r = float(np.exp(_r_rest.x[0]))
    _C_rest = OGSE_contrast_vs_g_rest(td, G_fit, G_fit, N_hi, N_lo, _tc_r, 1.0, _D0_r)
    entry["rest"] = {
        "tc": _tc_r, "D0": _D0_r, "C_curve": _C_rest,
        "cost": float(0.5 * np.sum((_C_rest - C_fit) ** 2)),
    }

    # ── tort model ────────────────────────────────────────────────────────────
    _D0_t = CF_TORT_D0_FIXED
    if CF_TORT_ALPHA_FIXED == "master":
        _al_t = _get_master_value(subj, roi, direction, CF_TORT_ALPHA_MASTER_COL)
    elif CF_TORT_ALPHA_FIXED is not None:
        _al_t = float(CF_TORT_ALPHA_FIXED)
    else:
        def _res_tort(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_t):
            r = OGSE_contrast_vs_g_tort(_td, _G, _G, N_hi, N_lo, p[0], 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_tort = least_squares(
            _res_tort, [CF_TORT_ALPHA_INIT],
            bounds=([CF_TORT_ALPHA_BOUNDS[0]], [CF_TORT_ALPHA_BOUNDS[1]]),
            method="trf", max_nfev=10000,
        )
        _al_t = float(_r_tort.x[0])
    _C_tort = OGSE_contrast_vs_g_tort(td, G_fit, G_fit, N_hi, N_lo, _al_t, 1.0, _D0_t)
    entry["tort"] = {
        "alpha": _al_t, "D0": _D0_t, "C_curve": _C_tort,
        "cost": float(0.5 * np.sum((_C_tort - C_fit) ** 2)),
    }

    # ── mixed model ───────────────────────────────────────────────────────────
    _D0_m       = CF_MIXED_D0_FIXED
    _tc_m_fixed = float(CF_MIXED_TC_FIXED) if CF_MIXED_TC_FIXED is not None else None
    if CF_MIXED_ALPHA_FIXED == "master":
        _al_m_fixed = _get_master_value(subj, roi, direction, CF_MIXED_ALPHA_MASTER_COL)
    elif CF_MIXED_ALPHA_FIXED is not None:
        _al_m_fixed = float(CF_MIXED_ALPHA_FIXED)
    else:
        _al_m_fixed = None
    _x0_m, _lo_m, _hi_m = [], [], []
    if _tc_m_fixed is None:
        _x0_m.append(np.log(CF_MIXED_TC_INIT))
        _lo_m.append(np.log(CF_MIXED_TC_BOUNDS[0])); _hi_m.append(np.log(CF_MIXED_TC_BOUNDS[1]))
    if _al_m_fixed is None:
        _x0_m.append(CF_MIXED_ALPHA_INIT)
        _lo_m.append(CF_MIXED_ALPHA_BOUNDS[0]); _hi_m.append(CF_MIXED_ALPHA_BOUNDS[1])
    if _x0_m:
        def _res_mixed(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_m,
                       _tcf=_tc_m_fixed, _alf=_al_m_fixed):
            _i  = 0
            _tc = np.exp(p[_i]) if _tcf is None else _tcf; _i += int(_tcf is None)
            _al = p[_i]          if _alf is None else _alf
            r   = OGSE_contrast_vs_g_mixed(_td, _G, _G, N_hi, N_lo, _tc, _al, 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_mixed = least_squares(
            _res_mixed, _x0_m, bounds=(_lo_m, _hi_m), method="trf", max_nfev=10000,
        )
        _i    = 0
        _tc_m = float(np.exp(_r_mixed.x[_i])) if _tc_m_fixed is None else _tc_m_fixed
        _i   += int(_tc_m_fixed is None)
        _al_m = float(_r_mixed.x[_i])           if _al_m_fixed is None else _al_m_fixed
    else:
        _tc_m = _tc_m_fixed if _tc_m_fixed is not None else CF_MIXED_TC_INIT
        _al_m = _al_m_fixed if _al_m_fixed is not None else CF_MIXED_ALPHA_INIT
    _C_mixed = OGSE_contrast_vs_g_mixed(td, G_fit, G_fit, N_hi, N_lo, _tc_m, _al_m, 1.0, _D0_m)
    entry["mixed"] = {
        "tc": _tc_m, "alpha": _al_m, "D0": _D0_m, "C_curve": _C_mixed,
        "cost": float(0.5 * np.sum((_C_mixed - C_fit) ** 2)),
    }

    contrast_fit_store[key] = entry

    print(
        f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  "
        f"rest tc={entry['rest']['tc']:.3f} ms  "
        f"tort α={entry['tort']['alpha']:.3f}  "
        f"mixed tc={entry['mixed']['tc']:.3f} ms α={entry['mixed']['alpha']:.3f}"
    )
    cf_rows.append(dict(
        subj=subj, roi=roi, direction=direction, td_ms=td,
        free_cost   =entry["free"]["cost"],
        rest_tc_ms  =entry["rest"]["tc"],    rest_cost  =entry["rest"]["cost"],
        tort_alpha  =entry["tort"]["alpha"], tort_cost  =entry["tort"]["cost"],
        mixed_tc_ms =entry["mixed"]["tc"],   mixed_alpha=entry["mixed"]["alpha"],
        mixed_cost  =entry["mixed"]["cost"],
    ))

cf_df = pd.DataFrame(cf_rows)
_cf_xlsx = OUT_DIR / CF_OUT_SUBDIR / "contrast_fit_results.xlsx"
cf_df.to_excel(_cf_xlsx, index=False)
print(f"\nSaved {len(cf_df)} rows → {_cf_xlsx}")
cf_df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT CONTRAST FITS  —  per (subj, roi, td)
# One figure per (subj, roi, td); columns = directions
# Data: corr_contrast values at N_hi G points (filled ●) and N_lo G points (empty ○)
# Fits: free (C0), rest (C1), tort (C2), mixed (C3)
# ══════════════════════════════════════════════════════════════════════════════

_cf_subjs = sorted(set(k[0] for k in contrast_fit_store))
_cf_rois  = sorted(set(k[1] for k in contrast_fit_store))
_cf_tds   = sorted(set(k[3] for k in contrast_fit_store))

for subj in _cf_subjs:
    for roi in _cf_rois:
        for td in _cf_tds:
            dirs_here = [d for d in DIRECTIONS if (subj, roi, d, td) in contrast_fit_store]
            if not dirs_here:
                continue
            fig, axes = plt.subplots(
                1, len(dirs_here),
                figsize=(4.5 * len(dirs_here), 4),
                sharey=True, squeeze=False,
            )
            axes = axes[0]

            for ax, direction in zip(axes, dirs_here):
                entry = contrast_fit_store[(subj, roi, direction, td)]
                G_c   = entry["G_common"]

                # ── data markers (values of the corr contrast at actual G points) ──
                ax.scatter(entry["G_hi"], entry["C_hi"],
                           color="k", s=20, zorder=5, label=f"N={N_hi} pts")
                ax.scatter(entry["G_lo"], entry["C_lo"],
                           facecolors="none", edgecolors="k", s=60, zorder=5,
                           label=f"N={N_lo} pts")

                # ── model fit lines ───────────────────────────────────────────
                ax.plot(G_c, entry["free"]["C_curve"],  color="C0", lw=1.5, label="free")
                ax.plot(G_c, entry["rest"]["C_curve"],  color="C1", lw=1.5,
                        label=f"rest  tc={entry['rest']['tc']:.2f} ms")
                ax.plot(G_c, entry["tort"]["C_curve"],  color="C2", lw=1.5,
                        label=f"tort  α={entry['tort']['alpha']:.3f}")
                ax.plot(G_c, entry["mixed"]["C_curve"], color="C3", lw=1.5,
                        label=f"mixed  tc={entry['mixed']['tc']:.2f} ms, α={entry['mixed']['alpha']:.3f}")

                ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                ax.set_title(direction, fontsize=9)
                ax.set_xlabel(G_LABEL)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                ax.legend(fontsize=6, loc="best")

            axes[0].set_ylabel(f"ΔS (N={N_hi}−N={N_lo}) [a.u.]")
            fig.suptitle(
                f"{subj}  |  {roi}  |  td={td:.0f} ms  —  contrast fits", fontsize=10
            )
            fig.tight_layout()
            fig.savefig(
                OUT_DIR / CF_OUT_SUBDIR / f"contrast_fit_{subj}_{roi}_{td:.0f}.png",
                dpi=120,
            )
            plt.close(fig)

print("Contrast fit plots saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTRAST CURVE FITTING (RAW) — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# Fitting target: raw_contrast_store[(subj,roi,dir,td)]
#                 sqrt( (f1·S1+f2·S2)² + (RN/M0_N)² ) at N_hi and N_lo, then subtracted.
# RN/M0 uses each N's own M0 = sqrt(S0²−RN²), matching cell 11.
# M0 = 1 in model calls (M0 absorbed into rn_hi / rn_lo).  D0 always fixed.
# ══════════════════════════════════════════════════════════════════════════════

CF_RAW_OUT_SUBDIR = "contrast_fits_raw"  # sub-folder inside OUT_DIR

# ── Free model ────────────────────────────────────────────────────────────────
CF_RAW_FREE_D0_FIXED   = D0_FIXED   # m²/ms

# ── Rest model ────────────────────────────────────────────────────────────────
CF_RAW_REST_TC_INIT   = 2.0         # ms
CF_RAW_REST_TC_FIXED  = None        # None → fitted; float → pinned
CF_RAW_REST_TC_BOUNDS = (0.05, 1000.0)
CF_RAW_REST_D0_FIXED  = D0_FIXED

# ── Tort model ────────────────────────────────────────────────────────────────
CF_RAW_TORT_ALPHA_INIT       = 0.1
CF_RAW_TORT_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_RAW_TORT_ALPHA_BOUNDS     = (0.0, 1.0)
CF_RAW_TORT_ALPHA_MASTER_COL = "alpha_macro"
CF_RAW_TORT_D0_FIXED         = D0_FIXED

# ── Mixed model ───────────────────────────────────────────────────────────────
CF_RAW_MIXED_TC_INIT          = 2.0
CF_RAW_MIXED_TC_FIXED         = None
CF_RAW_MIXED_TC_BOUNDS        = (0.05, 1000.0)
CF_RAW_MIXED_ALPHA_INIT       = 0.1
CF_RAW_MIXED_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_RAW_MIXED_ALPHA_BOUNDS     = (0.0, 1.0)
CF_RAW_MIXED_ALPHA_MASTER_COL = "alpha_macro"
CF_RAW_MIXED_D0_FIXED         = D0_FIXED

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT CONTRAST CURVES (RAW) — 4 SINGLE-COMPONENT MODELS
# Fitting target: raw_contrast_store  (includes Rician noise term per N)
# Model:  sqrt(S_model(N_hi)² + (RN/M0_hi)²) − sqrt(S_model(N_lo)² + (RN/M0_lo)²)
# M0 = 1 in signal calls; RN normalised by each N's own M0 = sqrt(S0²−RN²).
# D0 fixed per model.  tc fitted in log-space; alpha linear.
# ══════════════════════════════════════════════════════════════════════════════

from models.model_fitting import (
    M_ogse_free, M_ogse_rest, M_ogse_tort, M_ogse_mixed,
)

(OUT_DIR / CF_RAW_OUT_SUBDIR).mkdir(parents=True, exist_ok=True)

# ── raw-contrast wrappers: sqrt(S_hi² + rn_hi²) − sqrt(S_lo² + rn_lo²) ──────

def _Craw_free(td, G, N_hi, N_lo, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_free(td, G, N_hi, td / N_hi, M0, D0)
    S_lo = M_ogse_free(td, G, N_lo, td / N_lo, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_rest(td, G, N_hi, N_lo, tc, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_rest(td, G, N_hi, td / N_hi, tc, M0, D0)
    S_lo = M_ogse_rest(td, G, N_lo, td / N_lo, tc, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_tort(td, G, N_hi, N_lo, alpha, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_tort(td, G, N_hi, td / N_hi, alpha, M0, D0)
    S_lo = M_ogse_tort(td, G, N_lo, td / N_lo, alpha, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_mixed(td, G, N_hi, N_lo, tc, alpha, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_mixed(td, G, N_hi, td / N_hi, tc, alpha, M0, D0)
    S_lo = M_ogse_mixed(td, G, N_lo, td / N_lo, tc, alpha, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

# ── fitting loop ──────────────────────────────────────────────────────────────

contrast_raw_fit_store = {}
cf_raw_rows = []

for key in sorted(raw_contrast_store):
    subj, roi, direction, td = key
    G_fit = contrast_G_store[key]
    C_fit = raw_contrast_store[key]

    # ── circle markers: raw contrast values at actual data G points ───────────
    fits_td   = fit_store.get((subj, roi, direction), {}).get("fits", {}).get(td, {})
    G_hi_data = fits_td[N_hi]["G"] if N_hi in fits_td else np.array([])
    G_lo_data = fits_td[N_lo]["G"] if N_lo in fits_td else np.array([])
    C_at_Ghi  = np.interp(G_hi_data, G_fit, C_fit)
    C_at_Glo  = np.interp(G_lo_data, G_fit, C_fit)

    # ── per-N noise normalisation: RN / M0_N, matching cell 11 ────────────────
    rn    = fits_td[N_hi].get("RN", 0.0)
    rn_hi = rn / fits_td[N_hi]["M0"] if fits_td.get(N_hi, {}).get("M0", 0) > 0 else 0.0
    rn_lo = rn / fits_td[N_lo]["M0"] if fits_td.get(N_lo, {}).get("M0", 0) > 0 else 0.0

    entry = {
        "G_common": G_fit, "C_raw": C_fit,
        "G_hi": G_hi_data, "C_hi": C_at_Ghi,
        "G_lo": G_lo_data, "C_lo": C_at_Glo,
    }

    # ── free model ────────────────────────────────────────────────────────────
    _D0_f   = CF_RAW_FREE_D0_FIXED
    _C_free = _Craw_free(td, G_fit, N_hi, N_lo, 1.0, _D0_f, rn_hi, rn_lo)
    entry["free"] = {
        "D0": _D0_f, "C_curve": _C_free,
        "cost": float(0.5 * np.sum((_C_free - C_fit) ** 2)),
    }

    # ── rest model ────────────────────────────────────────────────────────────
    _D0_r = CF_RAW_REST_D0_FIXED
    if CF_RAW_REST_TC_FIXED is not None:
        _tc_r = float(CF_RAW_REST_TC_FIXED)
    else:
        def _res_rest(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_r,
                      _rn_hi=rn_hi, _rn_lo=rn_lo):
            r = _Craw_rest(_td, _G, N_hi, N_lo, np.exp(p[0]), 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_rest = least_squares(
            _res_rest, [np.log(CF_RAW_REST_TC_INIT)],
            bounds=([np.log(CF_RAW_REST_TC_BOUNDS[0])], [np.log(CF_RAW_REST_TC_BOUNDS[1])]),
            method="trf", max_nfev=10000,
        )
        _tc_r = float(np.exp(_r_rest.x[0]))
    _C_rest = _Craw_rest(td, G_fit, N_hi, N_lo, _tc_r, 1.0, _D0_r, rn_hi, rn_lo)
    entry["rest"] = {
        "tc": _tc_r, "D0": _D0_r, "C_curve": _C_rest,
        "cost": float(0.5 * np.sum((_C_rest - C_fit) ** 2)),
    }

    # ── tort model ────────────────────────────────────────────────────────────
    _D0_t = CF_RAW_TORT_D0_FIXED
    if CF_RAW_TORT_ALPHA_FIXED == "master":
        _al_t = _get_master_value(subj, roi, direction, CF_RAW_TORT_ALPHA_MASTER_COL)
    elif CF_RAW_TORT_ALPHA_FIXED is not None:
        _al_t = float(CF_RAW_TORT_ALPHA_FIXED)
    else:
        def _res_tort(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_t,
                      _rn_hi=rn_hi, _rn_lo=rn_lo):
            r = _Craw_tort(_td, _G, N_hi, N_lo, p[0], 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_tort = least_squares(
            _res_tort, [CF_RAW_TORT_ALPHA_INIT],
            bounds=([CF_RAW_TORT_ALPHA_BOUNDS[0]], [CF_RAW_TORT_ALPHA_BOUNDS[1]]),
            method="trf", max_nfev=10000,
        )
        _al_t = float(_r_tort.x[0])
    _C_tort = _Craw_tort(td, G_fit, N_hi, N_lo, _al_t, 1.0, _D0_t, rn_hi, rn_lo)
    entry["tort"] = {
        "alpha": _al_t, "D0": _D0_t, "C_curve": _C_tort,
        "cost": float(0.5 * np.sum((_C_tort - C_fit) ** 2)),
    }

    # ── mixed model ───────────────────────────────────────────────────────────
    _D0_m       = CF_RAW_MIXED_D0_FIXED
    _tc_m_fixed = float(CF_RAW_MIXED_TC_FIXED) if CF_RAW_MIXED_TC_FIXED is not None else None
    if CF_RAW_MIXED_ALPHA_FIXED == "master":
        _al_m_fixed = _get_master_value(subj, roi, direction, CF_RAW_MIXED_ALPHA_MASTER_COL)
    elif CF_RAW_MIXED_ALPHA_FIXED is not None:
        _al_m_fixed = float(CF_RAW_MIXED_ALPHA_FIXED)
    else:
        _al_m_fixed = None
    _x0_m, _lo_m, _hi_m = [], [], []
    if _tc_m_fixed is None:
        _x0_m.append(np.log(CF_RAW_MIXED_TC_INIT))
        _lo_m.append(np.log(CF_RAW_MIXED_TC_BOUNDS[0])); _hi_m.append(np.log(CF_RAW_MIXED_TC_BOUNDS[1]))
    if _al_m_fixed is None:
        _x0_m.append(CF_RAW_MIXED_ALPHA_INIT)
        _lo_m.append(CF_RAW_MIXED_ALPHA_BOUNDS[0]); _hi_m.append(CF_RAW_MIXED_ALPHA_BOUNDS[1])
    if _x0_m:
        def _res_mixed(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_m,
                       _tcf=_tc_m_fixed, _alf=_al_m_fixed,
                       _rn_hi=rn_hi, _rn_lo=rn_lo):
            _i  = 0
            _tc = np.exp(p[_i]) if _tcf is None else _tcf; _i += int(_tcf is None)
            _al = p[_i]          if _alf is None else _alf
            r   = _Craw_mixed(_td, _G, N_hi, N_lo, _tc, _al, 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_mixed = least_squares(
            _res_mixed, _x0_m, bounds=(_lo_m, _hi_m), method="trf", max_nfev=10000,
        )
        _i    = 0
        _tc_m = float(np.exp(_r_mixed.x[_i])) if _tc_m_fixed is None else _tc_m_fixed
        _i   += int(_tc_m_fixed is None)
        _al_m = float(_r_mixed.x[_i])           if _al_m_fixed is None else _al_m_fixed
    else:
        _tc_m = _tc_m_fixed if _tc_m_fixed is not None else CF_RAW_MIXED_TC_INIT
        _al_m = _al_m_fixed if _al_m_fixed is not None else CF_RAW_MIXED_ALPHA_INIT
    _C_mixed = _Craw_mixed(td, G_fit, N_hi, N_lo, _tc_m, _al_m, 1.0, _D0_m, rn_hi, rn_lo)
    entry["mixed"] = {
        "tc": _tc_m, "alpha": _al_m, "D0": _D0_m, "C_curve": _C_mixed,
        "cost": float(0.5 * np.sum((_C_mixed - C_fit) ** 2)),
    }

    contrast_raw_fit_store[key] = entry

    print(
        f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  "
        f"rest tc={entry['rest']['tc']:.3f} ms  "
        f"tort α={entry['tort']['alpha']:.3f}  "
        f"mixed tc={entry['mixed']['tc']:.3f} ms α={entry['mixed']['alpha']:.3f}"
    )
    cf_raw_rows.append(dict(
        subj=subj, roi=roi, direction=direction, td_ms=td,
        free_cost   =entry["free"]["cost"],
        rest_tc_ms  =entry["rest"]["tc"],    rest_cost  =entry["rest"]["cost"],
        tort_alpha  =entry["tort"]["alpha"], tort_cost  =entry["tort"]["cost"],
        mixed_tc_ms =entry["mixed"]["tc"],   mixed_alpha=entry["mixed"]["alpha"],
        mixed_cost  =entry["mixed"]["cost"],
    ))

cf_raw_df = pd.DataFrame(cf_raw_rows)
_cf_raw_xlsx = OUT_DIR / CF_RAW_OUT_SUBDIR / "contrast_raw_fit_results.xlsx"
cf_raw_df.to_excel(_cf_raw_xlsx, index=False)
print(f"\nSaved {len(cf_raw_df)} rows → {_cf_raw_xlsx}")
cf_raw_df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT CONTRAST FITS (RAW)  —  per (subj, roi, td)
# One figure per (subj, roi, td); columns = directions
# Data: raw_contrast values at N_hi G points (filled ●) and N_lo G points (empty ○)
# Fits: free (C0), rest (C1), tort (C2), mixed (C3)
# ══════════════════════════════════════════════════════════════════════════════

_cf_raw_subjs = sorted(set(k[0] for k in contrast_raw_fit_store))
_cf_raw_rois  = sorted(set(k[1] for k in contrast_raw_fit_store))
_cf_raw_tds   = sorted(set(k[3] for k in contrast_raw_fit_store))

for subj in _cf_raw_subjs:
    for roi in _cf_raw_rois:
        for td in _cf_raw_tds:
            dirs_here = [d for d in DIRECTIONS if (subj, roi, d, td) in contrast_raw_fit_store]
            if not dirs_here:
                continue
            fig, axes = plt.subplots(
                1, len(dirs_here),
                figsize=(4.5 * len(dirs_here), 4),
                sharey=True, squeeze=False,
            )
            axes = axes[0]

            for ax, direction in zip(axes, dirs_here):
                entry = contrast_raw_fit_store[(subj, roi, direction, td)]
                G_c   = entry["G_common"]

                # ── data markers (raw contrast at actual G points) ─────────────
                ax.scatter(entry["G_hi"], entry["C_hi"],
                           color="k", s=20, zorder=5, label=f"N={N_hi} pts")
                ax.scatter(entry["G_lo"], entry["C_lo"],
                           facecolors="none", edgecolors="k", s=60, zorder=5,
                           label=f"N={N_lo} pts")

                # ── model fit lines ───────────────────────────────────────────
                ax.plot(G_c, entry["free"]["C_curve"],  color="C0", lw=1.5, label="free")
                ax.plot(G_c, entry["rest"]["C_curve"],  color="C1", lw=1.5,
                        label=f"rest  tc={entry['rest']['tc']:.2f} ms")
                ax.plot(G_c, entry["tort"]["C_curve"],  color="C2", lw=1.5,
                        label=f"tort  α={entry['tort']['alpha']:.3f}")
                ax.plot(G_c, entry["mixed"]["C_curve"], color="C3", lw=1.5,
                        label=f"mixed  tc={entry['mixed']['tc']:.2f} ms, α={entry['mixed']['alpha']:.3f}")

                ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                ax.set_title(direction, fontsize=9)
                ax.set_xlabel(G_LABEL)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                ax.legend(fontsize=6, loc="best")

            axes[0].set_ylabel(f"ΔS_raw (N={N_hi}−N={N_lo}) [a.u.]")
            fig.suptitle(
                f"{subj}  |  {roi}  |  td={td:.0f} ms  —  raw contrast fits", fontsize=10
            )
            fig.tight_layout()
            fig.savefig(
                OUT_DIR / CF_RAW_OUT_SUBDIR / f"contrast_raw_fit_{subj}_{roi}_{td:.0f}.png",
                dpi=120,
            )
            plt.close(fig)

print("Raw contrast fit plots saved.")